In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.window import Window

In [0]:
%run /Workspace/DE_Project_Pipeline/1_setup/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text("catalog", "fmcg_de_project", "Catalog")
dbutils.widgets.text("data_source", "gross_price_non_aws_new", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

# base_path=f"s3://sportsbar-fmcg-de-project-viswadithya/{data_source}/*.csv"
# print(base_path)

In [0]:
# df = (
#     spark.read.format("csv")
#         .option("header", True)
#         .option("inferSchema", True)
#         .load(base_path)
#         .withColumn("read_timestamp", F.current_timestamp())
#         .select("*", "_metadata.file_name", "_metadata.file_size")
# )

In [0]:
# df.printSchema()

In [0]:
# display(df.limit(10))

In [0]:
# df.write\
#  .format("delta") \
#  .option("delta.enableChangeDataFeed", "true") \
#  .mode("overwrite") \
#  .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source};")
df_bronze=df_bronze.withColumn("ingestion_timestamp", F.current_timestamp())
# df_bronze.show(10)
display(df_bronze.count())


##Handling invalid records

In [0]:
df_invalid_products=df_bronze.filter(
        F.col("product_id").isNull() | 
        F.col("month").isNull() | 
        F.col("gross_price").isNull()
    )

In [0]:
df_invalid_products=df_invalid_products.withColumn(
    "rejection_reason",
    F.when(F.col("product_id").isNull(), "Missing product id")
    .when(F.col("month").isNull(), "Missing month")
    .when(F.col("gross_price").isNull(), "Missing gross_price")
)

display(df_invalid_products)

##Quarantine

In [0]:
df_invalid_products.write\
    .format("delta")\
        .mode("append")\
            .option("mergeSchema", True)\
                .saveAsTable(f"{catalog}.{silver_schema}.quarantine_{data_source}")


##valid records

In [0]:
df_valid_products=df_bronze.filter(
        (F.col("product_id").isNotNull() &
        F.col("month").isNotNull() &
        F.col("gross_price").isNotNull())
    )

In [0]:
Cdc_timestamp=F.date_sub(F.current_timestamp(), 2)
df_valid_products=df_valid_products.filter(F.col("ingestion_timestamp")>Cdc_timestamp)

display(df_valid_products)

In [0]:
print("Total Bronze records:", df_bronze.count())
print("Valid records:", df_valid_products.count())
print("Invalid records:", df_invalid_products.count())

As this is gross price, we will be having duplicate rcrods

In [0]:
# w=Window.partitionBy(F.col("customer_id")).orderBy(F.col("updated_at").desc())
# df_valid_products=df_valid_products.withColumn("rn",
#                                                F.row_number().over(w)
#                                                )

# display(df_valid_products)                                               

In [0]:
# df_valid_products=df_valid_products.filter(F.col("rn")==1)
# display(df_valid_products)

# df_silver=df_valid_products

In [0]:
df_bronze.select('month').distinct().show()

In [0]:

# 1️. Parse `month` from multiple possible formats
date_formats = ["yyyy/MM/dd", "dd/MM/yyyy", "yyyy-MM-dd", "dd-MM-yyyy"]

df_silver = df_bronze.withColumn(
    "month",
    F.coalesce(
        F.try_to_date(F.col("month"), "yyyy/MM/dd"),
        F.try_to_date(F.col("month"), "dd/MM/yyyy"),
        F.try_to_date(F.col("month"), "yyyy-MM-dd"),
        F.try_to_date(F.col("month"), "dd-MM-yyyy")
    )
)

In [0]:
df_silver.select('month').distinct().show()

In [0]:
df_silver.show(10)

In [0]:
# We are validating the gross_price column, converting only valid numeric values to double, fixing negative prices by making them positive, and replacing all non-numeric values with 0


df_silver = df_silver.withColumn(
    "gross_price",
    F.when(F.col("gross_price").rlike(r'^-?\d+(\.\d+)?$'), 
           F.when(F.col("gross_price").cast("double") < 0, -1 * F.col("gross_price").cast("double"))
            .otherwise(F.col("gross_price").cast("double")))
    .otherwise(0)
)

In [0]:
df_silver.show(10)

In [0]:
deleted_df=df_silver.filter(F.col("status")=="NON_ACTIVE")

display(deleted_df)

In [0]:
df_silver=df_silver.filter(
    F.col("status")=='ACTIVE'
)

display(df_silver)

In [0]:
# We enrich the silver dataset by performing an inner join with the products table to fetch the correct product_code for each product_id.

df_products = spark.table("fmcg_de_project.silver.products") 
df_joined = df_silver.join(df_products.select("product_id", "product_code"), on="product_id", how="inner")
df_joined = df_joined.select("product_id", "product_code", "month", "gross_price", "ingestion_timestamp")
df_silver=df_joined
df_joined.show(5)

In [0]:
silver_table = f"{catalog}.{silver_schema}.{data_source}"

In [0]:
if not spark.catalog.tableExists(silver_table):
    empty_df = spark.createDataFrame(
        [],
        """
        product_id STRING,
        product_code STRING,
        month STRING,
        gross_price STRING,
        ingestion_timestamp STRING,
        effective_start_date TIMESTAMP,
        effective_end_date TIMESTAMP,
        is_current BOOLEAN
        """
    )

    empty_df.write\
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", True)\
        .saveAsTable(silver_table)

In [0]:
silver_delta=DeltaTable.forName(
    spark,
    silver_table
)

current_active_df=(
        spark.table(silver_table)
        .filter(
            F.col("is_current")==True
        ).cache()
    )

In [0]:
deletions_to_process_df=deleted_df.alias("b").join(
    current_active_df.alias("s"),
    F.col("b.product_id")==F.col("s.product_id"),
    "inner"
).select(
    F.col("b.product_id"),
    F.col("b.ingestion_timestamp").alias("delete_timestamp")
)\
.dropDuplicates(["product_id"])

In [0]:
current_active_df.count()

In [0]:
silver_delta=DeltaTable.forName(spark, silver_table)

silver_delta.alias("s").merge(
    deletions_to_process_df.alias("d"),
    """
        s.product_id=d.product_id 
            and
        s.is_current=True
    """
).whenMatchedUpdate(
    condition="d.delete_timestamp>s.ingestion_timestamp",
    set={
        "is_current":"false",
        "effective_end_date": "d.delete_timestamp"
    }
).execute()

In [0]:
display(df_silver.limit(3))

In [0]:
silver_table=f"{catalog}.{silver_schema}.{data_source}"
bronze_df=df_silver

# window_spec=Window.partitionBy("product_id").orderBy(F.col("ingestion_timestamp").desc_nulls_last())

# bronze_df=bronze_df.withColumn("row_num", F.row_number().over(window_spec)).filter(F.col("row_num")==1).drop("row_num")


In [0]:
active_df=bronze_df.filter(
    F.upper(F.trim(F.col("status"))) == "ACTIVE"
)

deleted_df=bronze_df.filter(
    F.upper(F.trim(F.col("status"))) == "NON_ACTIVE"
)


In [0]:
display(s.limit(5))

In [0]:


silver_df=spark.table(silver_table).filter(F.col("is_current")==True)

joined_df=silver_df.alias("s").join(
    active_df.alias("b"),
    (F.col("s.product_id")==F.col("b.product_id")) & (F.col("s.month")==F.col("b.month")),
    "inner"
)




changed_df=joined_df.filter(
    ~(
            (F.col("s.product_id").eqNullSafe(F.col("b.product_id")))
            & (F.col("s.gross_price").eqNullSafe(F.col("b.gross_price")))
    )
).select("b.*")

new_df = active_df.alias("b").join(
    silver_df.select("product_id", "month").distinct().alias("s"),
    (F.col("b.product_id") == F.col("s.product_id"))
    & (F.col("b.month") == F.col("s.month")),
    "left_anti"
)

In [0]:

def prepare_new_version(df):
    return df.withColumn(
                "effective_start_date",
                F.col("ingestion_timestamp").cast("timestamp")
            )\
            .withColumn(
                "effective_end_date",
                F.lit(None).cast("timestamp")
            )\
            .withColumn(
                "is_current", F.lit("True")
            )

changed_versions = prepare_new_version(changed_df)
new_versions = prepare_new_version(new_df)

expire_rows = changed_versions.withColumn(
    "merge_key", F.col("product_id")
)

insert_changed_rows = changed_versions.withColumn(
    "merge_key",
    F.lit(None).cast(changed_versions.schema["product_id"].dataType)
)
insert_new_rows = new_versions.withColumn(
    "merge_key",
    F.lit(None).cast(new_versions.schema["product_id"].dataType)
)
staged_df = (
    expire_rows
    .unionByName(insert_changed_rows)
    .unionByName(insert_new_rows)
)


In [0]:
display(staged_df)

In [0]:

delta_silver= DeltaTable.forName(spark, silver_table)

delta_silver.alias("t").merge(
    staged_df.alias("s"),
    """
        t.product_id = s.merge_key
        AND t.month = s.month
        AND t.is_current = true
    """
).whenMatchedUpdate(
    condition="s.merge_key IS NOT NULL",
    set={
        "effective_end_date":"s.effective_start_date",
        "is_current":"false"
    }
).whenNotMatchedInsert(
    values={
        "product_id":"s.product_id",
        "product_code":"s.product_code",
        "month":"s.month",
        "gross_price":"s.gross_price",
        "ingestion_timestamp":"s.ingestion_timestamp",
        "is_current":"s.is_current",
        "effective_start_date":"s.effective_start_date",
        "effective_end_date":"s.effective_end_date"
    }
).execute()



In [0]:
# df_joined.write\
#  .format("delta") \
#  .option("delta.enableChangeDataFeed", "true")\
#  .option("mergeSchema", "true") \
#  .mode("overwrite") \
#  .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

In [0]:
# df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source};")

In [0]:
# # select only required columns
# df_gold = df_silver.select("product_code", "month", "gross_price")
# df_gold.show(5)

In [0]:
# df_gold.write\
#  .format("delta") \
#  .option("delta.enableChangeDataFeed", "true") \
#  .mode("overwrite") \
#  .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

In [0]:
# df_gold_price = spark.table("fmcg_de_project.gold.sb_dim_gross_price")
# df_gold_price.show(5)

In [0]:
# df_gold_price = (
#     df_gold_price
#     .withColumn("year", F.year("month"))
#     # 0 = non-zero price, 1 = zero price  ➜ non-zero comes first
#     .withColumn("is_zero", F.when(F.col("gross_price") == 0, 1).otherwise(0))
# )

# w = (
#     Window
#     .partitionBy("product_code", "year")
#     .orderBy(F.col("is_zero"), F.col("month").desc())
# )


# df_gold_latest_price = (
#     df_gold_price
#       .withColumn("rnk", F.row_number().over(w))
#       .filter(F.col("rnk") == 1)
# )


In [0]:
# display(df_gold_latest_price)

In [0]:
# ## Take required cols

# df_gold_latest_price = df_gold_latest_price.select("product_code", "year", "gross_price").withColumnRenamed("gross_price", "price_inr").select("product_code", "price_inr", "year")

# # change year to string
# df_gold_latest_price = df_gold_latest_price.withColumn("year", F.col("year").cast("string"))

# df_gold_latest_price.show(5)

In [0]:
# df_gold_latest_price.printSchema()

In [0]:
# delta_table = DeltaTable.forName(spark, "fmcg_de_project.gold.dim_gross_price")


# delta_table.alias("target").merge(
#     source=df_gold_latest_price.alias("source"),
#     condition="target.product_code = source.product_code"
# ).whenMatchedUpdate(
#     set={
#         "price_inr": "source.price_inr",
#         "year": "source.year"
#     }
# ).whenNotMatchedInsert(
#     values={
#         "product_code": "source.product_code",
#         "price_inr": "source.price_inr",
#         "year": "source.year"
#     }
# ).execute()